In [1]:
from os import listdir
from experiment_main import *

# ---------------------------------------------------------------------------------------
# Significance-test side channel (C7).
#
# deflated_sharpe needs two things that do not survive anywhere else: the SELECTED config's
# per-trade return series, and the sharpe of EVERY config that was searched (their spread
# sets the multiple-testing hurdle). naked_crossover computes the per-trade bets for all
# 475 configs on the validation window and get_crossover_results then discards them,
# keeping only the aggregates in dff_valid. This collects them instead.
#
# One record is appended per (crossover, system) run. Cleared by the caller before a run.
# ---------------------------------------------------------------------------------------
validation_bets = []


def get_crossover_results(
    df_test,
    df_valid,
    name_mode,
    target_lists,
    signal_line_list,
    trade_cost,
    balance,
    risk_per_trade,
    alpha_range,
    gamma_range,
    signal_line_if,
    signal_line_if_target,
    last_close,
):
    dff_test, current_bet_list_optim = naked_crossover(
        df_test,
        target_lists,
        signal_line_list,
        trade_cost,
        balance,
        risk_per_trade=risk_per_trade,
        signal_line_if=signal_line_if,
        signal_line_if_target=signal_line_if_target,
        last_close=last_close,
    )

    check_index_first = find_best_index(
        dff_test, 1, "net", name_mode, alpha_range, gamma_range
    )
    check_index_second = find_best_index(
        dff_test, 2, "net", name_mode, alpha_range, gamma_range
    )
    check_index_third, best_list = find_best_index(
        dff_test, 3, "net", name_mode, alpha_range, gamma_range
    )
    check_index_fourth = find_best_index(
        dff_test, 4, "net", name_mode, alpha_range, gamma_range
    )  # Genetic algorithm
    # The long/short rule above picks a config by "net". A long-only trader would pick
    # whichever config is best on LONG TRADES ALONE, so select again on "long_only_net".
    # The main experiment already does this (experiment_main.py ~3051) and splices it in
    # via check_list_maker; this notebook never did, which is why its "Long 2" column was
    # a duplicate of "Long 1" rather than a separate arm.
    check_index_first_long = find_best_index(
        dff_test, 1, "long_only_net", name_mode, alpha_range, gamma_range
    )
    check_index_second_long = find_best_index(
        dff_test, 2, "long_only_net", name_mode, alpha_range, gamma_range
    )
    check_index_third_long, best_list_long = find_best_index(
        dff_test, 3, "long_only_net", name_mode, alpha_range, gamma_range
    )
    check_index_fourth_long = find_best_index(
        dff_test, 4, "long_only_net", name_mode, alpha_range, gamma_range
    )  # Genetic algorithm

    base_histogram = dff_test[dff_test["his"] == f"{name_mode} a: 0"].index[0]

    # the base MACD row is the same config under either rule, so it is its own long-only arm
    long_indexes = [
        check_index_first_long,
        check_index_second_long,
        check_index_third_long,
        check_index_fourth_long,
        base_histogram,
    ]

    # print(check_index_first, check_index_second, check_index_third, check_index_fourth, base_histogram)

    dff_valid, current_bet_list_optim = naked_crossover(
        df_valid,
        target_lists,
        signal_line_list,
        trade_cost,
        balance,
        risk_per_trade=risk_per_trade,
        signal_line_if=signal_line_if,
        signal_line_if_target=signal_line_if_target,
        last_close=last_close,
    )

    # stash every config's per-trade bets on the validation window, plus which configs
    # the four rules and the base picked, so the significance tests can be run afterwards
    # without re-running the backtest
    validation_bets.append(
        {
            "name_mode": name_mode,
            "signal_line_if": signal_line_if,
            "signal_line_if_target": signal_line_if_target,
            "chosen": {
                "rule_1": dff_valid.iloc[check_index_first]["his"],
                "rule_2": dff_valid.iloc[check_index_second]["his"],
                "rule_3": dff_valid.iloc[check_index_third]["his"],
                "rule_4": dff_valid.iloc[check_index_fourth]["his"],
                "base": dff_valid.iloc[base_histogram]["his"],
            },
            "bets": current_bet_list_optim,
        }
    )

    return (
        dff_valid,
        check_index_first,
        check_index_second,
        check_index_third,
        check_index_fourth,
        base_histogram,
        long_indexes,
    )


def return_strategies_performance(
    nikkei_data,
    major_list,
    start_timestamp,
    data_part,
    strategies,
    target_lists,
    signal_line_list,
    trade_cost,
    balance,
    risk_per_trade,
    alpha_range,
    gamma_range,
    signal_line_if,
    signal_line_if_target,
    last_close,
):
    two_years = 24
    start_optimize = make_new_time(start_timestamp, -two_years)
    end_optimize = start_timestamp
    optimize_data = nikkei_data[start_optimize:end_optimize]
    sys_data_list = [
        [ [0 for i in range(14)],
        [0 for i in range(14)],
        [0 for i in range(14)],
        [0 for i in range(14)],
        [0 for i in range(14)] ],
        [ [0 for i in range(14)],
        [0 for i in range(14)],
        [0 for i in range(14)],
        [0 for i in range(14)],
        [0 for i in range(14)] ],
    ]
    best_sys_minor = []
    dff_results_list = []
    
    for i in range(2):
        strategy, target_list, sys_data, current_major_list = strategies[i], target_lists[i], sys_data_list[i], major_list[i]
        
        name_mode = get_name_mode(strategy)
        (
            dff,
            check_index_first,
            check_index_second,
            check_index_third,
            check_index_fourth,
            base_histogram,
            long_indexes,
        ) = get_crossover_results(
            optimize_data,
            data_part,
            name_mode,
            target_list,
            signal_line_list,
            trade_cost,
            balance,
            risk_per_trade,
            alpha_range,
            gamma_range,
            signal_line_if,
            signal_line_if_target,
            last_close,
        )

        add_df_to_list_detailed(
            current_major_list,
            dff,
            check_index_first,
            check_index_second,
            check_index_third,
            check_index_fourth,
            base_histogram,
        )
        # the major rows are summed over the three sub-periods, so the long-only arm has
        # to be accumulated rather than appended
        accumulate_long_only(current_major_list, dff, long_indexes)
        major_list[i] = current_major_list

        add_df_to_list_detailed(
            sys_data,
            dff,
            check_index_first,
            check_index_second,
            check_index_third,
            check_index_fourth,
            base_histogram,
        )
        # per-period rows are built fresh, so here the arm is appended
        extend_with_long_only(sys_data, dff, long_indexes)
        for j in range(len(sys_data)):
            print(sys_data[j], len(sys_data[j]))
        sys_data_list[i] = sys_data
        
        best_sys_minor.append(get_best_results(sys_data))
        
        dff_results_list.append([dff,
            check_index_first,
            check_index_second,
            check_index_third,
            check_index_fourth,
            base_histogram])

    return (
        dff_results_list,
        major_list,
        sys_data_list,
        best_sys_minor,
    )


def extend_with_long_only(output_list, value_df, long_indexes):
    """
    extend_with_long_only appends the genuine long-only arm to rows that are built fresh for one period, taking the figures from the config the LONG-ONLY rule selected rather than from the config the long/short rule selected

    Replaces `row.extend(row[-3:])`, which duplicated the three long-only columns already
    on the row and so made the reported "Long 2" identical to "Long 1" by construction.

    Arguments:
        output_list -- the five result rows for this period (first/second/third/fourth/base)
        value_df -- the validation results frame, whose columns 11:14 are long_only_net, long_only_bets and long_only_wins
        long_indexes -- the five row indexes chosen on "long_only_net"

    Returns:
        None -- extends each row of output_list in place
    """
    for position, long_index in enumerate(long_indexes):
        output_list[position].extend(list(value_df.iloc[long_index])[11:14])


def accumulate_long_only(output_list, value_df, long_indexes):
    """
    accumulate_long_only adds this period's long-only figures into rows that are SUMMED across the three sub-periods, which is why the major lists cannot use extend_with_long_only

    The major lists are initialised at 17 zeros and accumulated with += by
    add_df_to_list_detailed for the first 14 fields; this does the same for fields 14:17,
    the long-only arm. Each period contributes the config ITS OWN long-only rule selected.

    Arguments:
        output_list -- the five accumulating rows for one strategy
        value_df -- the validation results frame for this period
        long_indexes -- the five row indexes chosen on "long_only_net" for this period

    Returns:
        None -- accumulates into output_list in place
    """
    for position, long_index in enumerate(long_indexes):
        values = list(value_df.iloc[long_index])[11:14]
        for offset, value in enumerate(values):
            output_list[position][14 + offset] += value


def add_df_to_list_detailed(
    output_list,
    value_df,
    index_first,
    index_second,
    index_third,
    index_fourth,
    base_his,
):
    length_of_list = len(value_df.iloc[index_first]) - 1
    index_first_list = list(value_df.iloc[index_first])
    index_second_list = list(value_df.iloc[index_second])
    index_third_list = list(value_df.iloc[index_third])
    index_fourth_list = list(value_df.iloc[index_fourth])
    index_base_list = list(value_df.iloc[base_his])
    for i in range(length_of_list):
        output_list[0][i] += index_first_list[i]
        output_list[1][i] += index_second_list[i]
        output_list[2][i] += index_third_list[i]
        output_list[3][i] += index_fourth_list[i]
        output_list[4][i] += index_base_list[i]


def add_df_to_list_overall(output_list, value_df, index_first, base_his):
    length_of_list = len(value_df.iloc[index_first]) - 1
    index_first_list = list(value_df.iloc[index_first])
    index_base_list = list(value_df.iloc[base_his])
    for i in range(length_of_list):
        output_list[0][i] += index_first_list[i]
        output_list[1][i] += index_base_list[i]


def get_data_rule(rule_data, len_return_list):
    ticker_length = 1
    return_list = [[] for i in range(len_return_list)]

    [
        "net",  # 0
        "netbet",  # 1
        "netpercent",  # 2
        "bets",  # 3
        "wins",  # 4
        "longs",  # 5
        "long_wins",  # 6
        "long_profit",  # 7
        "shorts",  # 8
        "short_wins",  # 9
        "short_profit",  # 10
        "long_only_net",  # 11
        "long_only_bets",  # 12
        "long_only_wins",  # 13
        "long_only_net_other",  # 14
        "long_only_bets_other",  # 15
        "long_only_wins_other",  # 16
        "his",  # 17
    ]
    # [0'net' , 1'netbet' , 2'netpercent' , 3'bets' , 4'wins' , 5'longs' , 6'long_wins' , 7'long_profit' , 8'shorts' , 9'short_wins' , 10'short_profit',
    # 11"long_only_net", 12"long_only_bets", 13"long_only_wins", 14"long_only_net_other", 15"long_only_bets_other", 16"long_only_wins_other", 17'his']

    # avg return for ticker , bets , win/loss , avg long return for ticker , longs , long win ratio , avg short return , short win ratio,
    # 0 / tickers , 3 , 4/3 * % , 7 / tickers , 5 , 6 / 5 * % , 10 / tickers , 9 / 8 * %

    # average long only return for ticker, long only bets, long only win ratio, average other long return for ticker, other long bets, other long win ratio
    # 11 / tickers, 12, 13 / 12 * %, 14 / ticker, 15, 16 / 15 * %
    # print(len(obs_mode_data))
    # print(len(rule_data))
    for i in range(len(rule_data)):
        return_list[i].extend(
            [
                (rule_data[i][0]),  # / ticker_length)
                rule_data[i][3],
                (
                    (rule_data[i][4] * 100 / rule_data[i][3])
                    if rule_data[i][3] != 0
                    else 0
                ),
                (rule_data[i][7]),  # / ticker_length)
                rule_data[i][5],
                (
                    (rule_data[i][6] * 100 / rule_data[i][5])
                    if rule_data[i][5] != 0
                    else 0
                ),
                (rule_data[i][10]),  # / ticker_length)
                (
                    (rule_data[i][9] * 100 / rule_data[i][8])
                    if rule_data[i][8] != 0
                    else 0
                ),
                # Long 1 = the long leg of the config the LONG/SHORT rule picked
                (rule_data[i][11]),  # / ticker_length)
                rule_data[i][12],
                (
                    (rule_data[i][13] * 100 / rule_data[i][12])
                    if rule_data[i][12] != 0
                    else 0
                ),
                # Long 2 = the genuine long-only arm: the config the LONG-ONLY rule picked
                (rule_data[i][14]),  # / ticker_length)
                rule_data[i][15],
                (
                    (rule_data[i][16] * 100 / rule_data[i][15])
                    if rule_data[i][15] != 0
                    else 0
                ),
            ]
        )

    return return_list


def get_best_results(sys_crossover_list):
    best_results_index = 0
    for i in range(1, len(sys_crossover_list) - 1):
        # print(sys_crossover_list[i][0], sys_crossover_list[i-1][0])
        if sys_crossover_list[i][0] > sys_crossover_list[best_results_index][0]:
            best_results_index = i
    return sys_crossover_list[best_results_index]

def extend_major_list(sys_major_list):
    best_sys_major_list = []
    for current_sys_major in sys_major_list:
        for i in range(len(current_sys_major)):
            # fields 14:17 were accumulated by accumulate_long_only as each period ran, so
            # there is nothing to append here. This used to duplicate fields 11:13.
            print(current_sys_major[i], len(current_sys_major[i]))
        best_sys_major_list.append(get_best_results(current_sys_major))
    return best_sys_major_list

def table_data_generator(data, indexes, len_return_list):
    output_data = []
    for d in data:
        current_return_list = get_data_rule(d, len_return_list)
        output_data.extend(current_return_list)

    # naked_cols = ['net', 'netbet', 'netpercent', 'bets', 'wins', 'longs', 'long_wins', 'long_profit',
    # 'shorts', 'short_wins', 'short_profit', "long_only_net", "long_only_bets", "long_only_wins",
    # "long_only_net_other", "long_only_bets_other", "long_only_wins_other", 'his']

    tempdf = pd.DataFrame(
        output_data,
        columns=[
            "Return",
            "Transactions",
            "Win Ratio",
            "Long Return",
            "Long Transactions",
            "Long Win Ratio",
            "Short Return",
            "Short Win Ratio",
            "Long 1 Return",
            "Long 1 Transactions",
            "Long 1 Win Ratio",
            "Long 2 Return",
            "Long 2 Transactions",
            "Long 2 Win Ratio",
        ],
        index=indexes,
    )

    return tempdf


alpha_range = [
    -0.9,
    -0.8,
    -0.7,
    -0.6,
    -0.5,
    -0.4,
    -0.3,
    -0.2,
    -0.1,
    0,
    0.1,
    0.2,
    0.3,
    0.4,
    0.5,
    0.6,
    0.7,
    0.8,
    0.9,
]
gamma_range = [
    1,
    2,
    4,
    10,
    20,
    30,
    40,
    50,
    60,
    70,
    80,
    90,
    100,
    200,
    300,
    400,
    500,
    600,
    700,
    800,
    900,
    1000,
    10000,
    100000,
]
histogram_list, macd_list, signal_line_list = return_macd_lists(
    alpha_range, gamma_range
)
target_lists = [histogram_list, macd_list]

In [ ]:
prefix_tex_tables = "/media/calavera/1TB HDD/Projects/MACD Project/Paper/paper_ready/table_image/"

market = "nikkei"
prefix_text = f"./results/{market}"
file_name = f"insert_text_{market}.txt"
time_format = "1_day"

macd_params = (12, 26, 9)
# macd_params = (4, 22, 3)

# Kang traded front-month FUTURES. We now have both instruments and run the notebook once
# per series, because neither alone does the whole job:
#   "index"   -- nkx_d.csv, the Nikkei 225 cash index on the JPX calendar. Covers Kang's
#                full window at exactly his 2204 trading days, so it is the only series
#                that can match his whole-period and all three sub-period figures. Wrong
#                instrument: no basis, no roll.
#   "futures" -- Barchart's back-adjusted Nearby series for the Osaka yen contract,
#                2001-10-02 onward, which covers Kang's full window at exactly his 2204
#                days. We use a vendor continuous series here only because we first built
#                our own from 33 per-contract files and showed the two agree: over the
#                2,004 overlapping days their daily changes differ from ours by a median
#                of 0.0000, with 19 exceptions of which 17 are one-day roll-timing
#                differences on our own roll dates. Our stitched series and its roll log
#                stay on disk as the audit trail. (Our own build could not be used
#                directly because Barchart retains no expired Nikkei contract older than
#                2012, so it starts 2012-01-04 and could not validate before 2014-01.)
price_series = "index"
# price_series = "futures"

mode = 'a'

# the series goes into params_str so the two runs write to separate results folders
params_str = "_".join(str(val) for val in macd_params) + f"_{price_series}"
short_period, long_period, signal_period = macd_params

# every (parameters, window) combination writes into its own results folder, and the
# 11-config sweep is about to add eleven more per market, so create the folder instead of
# requiring it to already exist. `from os import listdir` in cell 0 does not bind `os`.
import os

os.makedirs(f"./results/{market}/{params_str}", exist_ok=True)

trade_cost = False
balance = 0
risk_per_trade = 100 / 100
last_close = True
strategies = ["MACDCROSS", "ZEROCROSS"]

prefix = f"./data/{market}/"

# nkx_d.csv is the INDEX on the JPX calendar: 2204 trading days over 2011-01-04..
# 2019-12-30, exactly Kang's stated count, and no bars on Japanese public holidays. The
# old nkf.csv is NOT a JPX series at all -- it trades through 20/20 Japanese holidays and
# only 8/15 US ones, and its daily changes correlate 0.52 with the index because the CME
# day settles 16:00 CT, about 06:00 JST the next morning. It is the CME contract and is
# not used. It cannot fill the futures series' 2011 gap either: over 2011 alone it has 257
# bars against the index's 245, trades on 10/10 Japanese holidays, and correlates 0.41.
nikkei_data = pd.read_csv(
    prefix + ("nkx_d.csv" if price_series == "index"
              else "nikkei_futures_nearby_backadjusted.csv"),
    index_col="Date",
    parse_dates=True,
    sep=",",
    usecols=["Date", "Close", "Open", "High", "Low"],
)

nikkei_data.dropna(inplace=True)
nikkei_data.columns = list(map(lambda x: x.lower(), list(nikkei_data.columns)))
nikkei_data.sort_index(inplace=True)
nikkei_data.index.names = ["date"]

nikkei_data = calculate_indicators(
    nikkei_data,
    col="close",
    ema_mode="normal",
    short_period=short_period,
    long_period=long_period,
    signal_period=signal_period,
    alpha_range=alpha_range,
    gamma_range=gamma_range,
    # calculate_indicators used to clip every series it was handed to 2009-07-01 with a
    # hardcoded line. that left only 18 months of history before Kang's first check at
    # 2011-01-01, against the 24-month optimization window this notebook uses
    # (start_optimize = make_new_time(start_timestamp, -two_years)), so the first
    # optimization window was silently short by a quarter of its length. nkx_d.csv goes
    # back to 1914, so start three years early instead: 24 months of optimization plus a
    # year of slack for the EMAs to converge before any of it is used.
    start_date=pd.Timestamp("2008-01-01"),
)

# the engine's strategies execute at a 'fill' column; without it macd_crossover raises
# KeyError. KANG'S EXECUTION RULE: "a buy (or sell) order for one trading unit is executed
# at the closing price (index value) on the next day". The default fill on daily bars is
# the signal bar's OWN close, which is a day early. Measured over his window, the median
# |next close - this close| is 101 index points against results of 1,000-16,000 points over
# 50-175 round trips, so the displacement is material and does not reliably cancel.
add_fill_column(nikkei_data, mode="next_close")

nikkei_data

,open,high,low,close,MACD,SignalLine,Histogram,MACD a: -0.9,MACD Signal a: -0.9,Histogram a: -0.9,...,Histogram a: 0.9 g: 1000,MACD a: 0.9 g: 1000 times,MACD a: 0.9 g: 10000,MACD Signal a: 0.9 g: 10000,Histogram a: 0.9 g: 10000,MACD a: 0.9 g: 10000 times,MACD a: 0.9 g: 100000,MACD Signal a: 0.9 g: 100000,Histogram a: 0.9 g: 100000,MACD a: 0.9 g: 100000 times
2009-07-01,10000.0,10100.0,9982.0,10047.0,175.369106,153.018562,22.350544,257.740322,186.679127,71.061194,...,-5.212120,3288,172.364979,177.577099,-5.212120,3291,172.364979,177.577099,-5.212120,3294
2009-07-02,9878.0,9913.0,9707.0,9738.0,89.236191,121.127376,-31.891185,-62.417526,62.130801,-124.548327,...,-2.455854,3288,172.665392,175.121245,-2.455854,3291,172.665392,175.121245,-2.455854,3294
2009-07-03,9843.0,9858.0,9795.0,9803.0,59.294292,90.210834,-30.916542,168.834929,115.482865,53.352064,...,-5.399387,3288,164.322471,169.721858,-5.399387,3291,164.322471,169.721858,-5.399387,3294
2009-07-06,9680.0,9748.0,9640.0,9727.0,17.037624,53.624229,-36.586605,-119.579950,-2.048543,-117.531407,...,-7.951102,3288,153.819654,161.770756,-7.951102,3291,153.819654,161.770756,-7.951102,3294
2009-07-07,9668.0,9727.0,9527.0,9552.0,-61.486902,-3.931337,-57.555566,-9.203160,-5.625851,-3.577309,...,-10.814653,3288,140.141451,150.956103,-10.814653,3291,140.141451,150.956103,-10.814653,3294
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2019-12-24,23825.0,23830.0,23755.0,23780.0,207.528023,246.770914,-39.242891,260.534107,251.194798,9.339309,...,1.573617,3288,273.424872,271.851255,1.573617,3291,273.424872,271.851255,1.573617,3294
2019-12-26,23770.0,23960.0,23725.0,23955.0,232.069037,239.419976,-7.350938,206.450475,228.822636,-22.372161,...,-2.508034,3288,266.835187,269.343221,-2.508034,3291,266.835187,269.343221,-2.508034,3294
2019-12-27,23910.0,23960.0,23770.0,23795.0,187.354322,213.387149,-26.032827,170.167785,199.495210,-29.327426,...,-2.992325,3288,263.358572,266.350897,-2.992325,3291,263.358572,266.350897,-2.992325,3294
2019-12-30,23830.0,23830.0,23415.0,23430.0,42.080893,127.734021,-85.653128,-73.197309,63.148950,-136.346260,...,-5.296375,3288,255.758147,261.054522,-5.296375,3291,255.758147,261.054522,-5.296375,3294


In [3]:
two_years = 24
# zero_check = pd.Timestamp("2008-01-01")
if price_series == "index":
    # Kang's own three sub-periods, which the index covers in full
    first_check = pd.Timestamp("2011-01-01")
    second_check = pd.Timestamp("2014-01-01")
    third_check = pd.Timestamp("2017-01-01")
    fourth_check = pd.Timestamp("2020-01-01")
else:
    # Barchart's Nearby series looks like it reaches 2001, but it has a HOLE: 2008 and
    # 2009 are absent entirely and 2010 carries 11 rows. Full density resumes in 2011.
    # Kang's first sub-period would need to optimise over 2009-2011, which lands in that
    # hole -- an earlier attempt at his own boundaries silently optimised period 1 on
    # ELEVEN bars. So the futures validation starts in 2013, the first date with a full
    # 24-month optimisation window behind it (2011-01..2013-01). This covers 2013-2019,
    # seven of Kang's nine years; the boundaries are ours, not his, so compare on the
    # whole-period figure and say so.
    first_check = pd.Timestamp("2013-01-01")
    second_check = pd.Timestamp("2015-01-01")
    third_check = pd.Timestamp("2017-01-01")
    fourth_check = pd.Timestamp("2020-01-01")

# checks_list = [zero_check, first_check, second_check, third_check, fourth_check]
checks_list = [first_check, second_check, third_check, fourth_check]

# zero_part = nikkei_data[zero_check:first_check]
first_part = nikkei_data[first_check:second_check]
second_part = nikkei_data[second_check:third_check]
third_part = nikkei_data[third_check:fourth_check]

# df_list = [zero_part, first_part, second_part, third_part]
df_list = [first_part, second_part, third_part]

first_part

,open,high,low,close,MACD,SignalLine,Histogram,MACD a: -0.9,MACD Signal a: -0.9,Histogram a: -0.9,...,Histogram a: 0.9 g: 1000,MACD a: 0.9 g: 1000 times,MACD a: 0.9 g: 10000,MACD Signal a: 0.9 g: 10000,Histogram a: 0.9 g: 10000,MACD a: 0.9 g: 10000 times,MACD a: 0.9 g: 100000,MACD Signal a: 0.9 g: 100000,Histogram a: 0.9 g: 100000,MACD a: 0.9 g: 100000 times
2011-01-04,10375.0,10460.0,10345.0,10345.0,67.253095,67.507812,-0.254717,125.740768,86.225548,39.515221,...,-9.390806,3288,142.888980,152.279787,-9.390806,3291,142.888980,152.279787,-9.390806,3294
2011-01-05,10420.0,10555.0,10360.0,10545.0,141.677325,104.592569,37.084757,156.020227,121.122887,34.897339,...,-8.477198,3288,135.325392,143.802589,-8.477198,3291,135.325392,143.802589,-8.477198,3294
2011-01-06,10540.0,10585.0,10495.0,10520.0,169.694866,137.143717,32.551149,182.002042,151.562465,30.439577,...,-3.921002,3288,135.960585,139.881587,-3.921002,3291,135.960585,139.881587,-3.921002,3294
2011-01-07,10545.0,10595.0,10455.0,10505.0,174.445524,155.794621,18.650903,167.644657,159.603561,8.041096,...,-0.273787,3288,139.334013,139.607800,-0.273787,3291,139.334013,139.607800,-0.273787,3294
2011-01-10,10545.0,10545.0,10400.0,10480.0,163.154301,159.474461,3.679840,159.112980,159.358270,-0.245291,...,1.618682,3288,142.845164,141.226482,1.618682,3291,142.845164,141.226482,1.618682,3294
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2013-12-24,16047.5,16050.5,15863.0,15882.5,336.689675,322.856101,13.833574,434.530118,358.751321,75.778797,...,6.935193,3288,253.540982,246.605789,6.935193,3291,253.540982,246.605789,6.935193,3294
2013-12-26,16212.5,16251.5,16197.5,16251.5,423.534099,373.195100,50.338999,413.637682,386.194501,27.443181,...,7.625031,3288,261.855851,254.230820,7.625031,3291,261.855851,254.230820,7.625031,3294
2013-12-27,16247.5,16312.5,16063.0,16292.5,469.212902,421.204001,48.008901,519.230599,452.712550,66.518049,...,11.896428,3288,278.023676,266.127248,11.896428,3291,278.023676,266.127248,11.896428,3294
2013-12-30,16290.5,16333.0,16183.0,16291.5,477.603449,449.403725,28.199724,440.139014,446.425782,-6.286768,...,15.507675,3288,297.142599,281.634923,15.507675,3291,297.142599,281.634923,15.507675,3294


# SYSTEM 1 

In [4]:
signal_line_if = False
signal_line_if_target = False
sys1_major = [
    [ [0 for i in range(17)],
    [0 for i in range(17)],
    [0 for i in range(17)],
    [0 for i in range(17)],
    [0 for i in range(17)] ],
    [ [0 for i in range(17)],
    [0 for i in range(17)],
    [0 for i in range(17)],
    [0 for i in range(17)],
    [0 for i in range(17)] ],
]

# first part

In [5]:
(
    dff_results_list,
    
    sys1_major,
    sys1_first,
    best_sys1_first,
    
) = return_strategies_performance(
    nikkei_data,
    
    sys1_major,
    first_check,
    first_part,
    
    strategies,
    target_lists,
    signal_line_list,
    trade_cost,
    balance,
    risk_per_trade,
    alpha_range,
    gamma_range,
    signal_line_if,
    signal_line_if_target,
    last_close,
)

i = 0
dff, check_index_first, check_index_second, check_index_third, check_index_fourth, base_histogram = dff_results_list[i]
dff.iloc[
    [
        check_index_first,
        check_index_second,
        check_index_third,
        check_index_fourth,
        base_histogram,
    ]
]

[np.float64(-1788.5), np.float64(-12.25), np.int64(0), np.int64(146), np.int64(47), np.int64(73), np.int64(29), np.float64(1786.5), np.int64(73), np.int64(18), np.float64(-3575.0), np.float64(1786.5), np.int64(73), np.int64(29), np.float64(1786.5), np.int64(73), np.int64(29)] 17
[np.float64(2068.5), np.float64(22.483695652173914), np.int64(0), np.int64(92), np.int64(36), np.int64(46), np.int64(21), np.float64(3662.5), np.int64(46), np.int64(15), np.float64(-1594.0), np.float64(3662.5), np.int64(46), np.int64(21), np.float64(3662.5), np.int64(46), np.int64(21)] 17
[np.float64(2068.5), np.float64(22.483695652173914), np.int64(0), np.int64(92), np.int64(36), np.int64(46), np.int64(21), np.float64(3662.5), np.int64(46), np.int64(15), np.float64(-1594.0), np.float64(3662.5), np.int64(46), np.int64(21), np.float64(3662.5), np.int64(46), np.int64(21)] 17
[np.float64(2877.5), np.float64(31.277173913043477), np.int64(0), np.int64(92), np.int64(43), np.int64(46), np.int64(24), np.float64(4141.5)

,net,netbet,netpercent,bets,wins,longs,long_wins,long_profit,shorts,short_wins,short_profit,long_only_net,long_only_bets,long_only_wins,his
10,-1788.5,-12.250000,0,146,47,73,29,1786.5,73,18,-3575.0,1786.5,73,29,Histogram a: 0.1
16,2068.5,22.483696,0,92,36,46,21,3662.5,46,15,-1594.0,3662.5,46,21,Histogram a: 0.7
16,2068.5,22.483696,0,92,36,46,21,3662.5,46,15,-1594.0,3662.5,46,21,Histogram a: 0.7
405,2877.5,31.277174,0,92,43,46,24,4141.5,46,19,-1264.0,4141.5,46,24,Histogram a: 0.7 g: 4
9,-2150.5,-13.785256,0,156,49,78,29,1605.5,78,20,-3756.0,1605.5,78,29,Histogram a: 0


# second part

In [6]:
(
    dff_results_list,
    
    sys1_major,
    sys1_second,
    best_sys1_second,
    
) = return_strategies_performance(
    nikkei_data,
    
    sys1_major,
    second_check,
    second_part,
    
    strategies,
    target_lists,
    signal_line_list,
    trade_cost,
    balance,
    risk_per_trade,
    alpha_range,
    gamma_range,
    signal_line_if,
    signal_line_if_target,
    last_close,
)

i = 0
dff, check_index_first, check_index_second, check_index_third, check_index_fourth, base_histogram = dff_results_list[i]
dff.iloc[
    [
        check_index_first,
        check_index_second,
        check_index_third,
        check_index_fourth,
        base_histogram,
    ]
]

[np.float64(-1257.5), np.float64(-7.762345679012346), np.int64(0), np.int64(162), np.int64(72), np.int64(81), np.int64(38), np.float64(1017.5), np.int64(81), np.int64(34), np.float64(-2275.0), np.float64(1017.5), np.int64(81), np.int64(38), np.float64(1017.5), np.int64(81), np.int64(38)] 17
[np.float64(2551.5), np.float64(34.479729729729726), np.int64(0), np.int64(74), np.int64(25), np.int64(37), np.int64(13), np.float64(2922.0), np.int64(37), np.int64(12), np.float64(-370.5), np.float64(2922.0), np.int64(37), np.int64(13), np.float64(2922.0), np.int64(37), np.int64(13)] 17
[np.float64(2551.5), np.float64(34.479729729729726), np.int64(0), np.int64(74), np.int64(25), np.int64(37), np.int64(13), np.float64(2922.0), np.int64(37), np.int64(12), np.float64(-370.5), np.float64(2922.0), np.int64(37), np.int64(13), np.float64(2922.0), np.int64(37), np.int64(13)] 17
[np.float64(-6998.5), np.float64(-64.80092592592592), np.int64(0), np.int64(108), np.int64(39), np.int64(54), np.int64(20), np.flo

,net,netbet,netpercent,bets,wins,longs,long_wins,long_profit,shorts,short_wins,short_profit,long_only_net,long_only_bets,long_only_wins,his
211,-1257.5,-7.762346,0,162,72,81,38,1017.5,81,34,-2275.0,1017.5,81,38,Histogram a: -0.1 g: 1
17,2551.5,34.479730,0,74,25,37,13,2922.0,37,12,-370.5,2922.0,37,13,Histogram a: 0.8
17,2551.5,34.479730,0,74,25,37,13,2922.0,37,12,-370.5,2922.0,37,13,Histogram a: 0.8
370,-6998.5,-64.800926,0,108,39,54,20,-1853.0,54,19,-5145.5,-1853.0,54,20,Histogram a: 0.5 g: 400
9,2082.5,13.611111,0,153,56,77,32,2856.5,76,24,-774.0,2856.5,77,32,Histogram a: 0


# third part

In [7]:
(
    dff_results_list,
    
    sys1_major,
    sys1_third,
    best_sys1_third,

) = return_strategies_performance(
    nikkei_data,
    
    sys1_major,
    third_check,
    third_part,
    
    strategies,
    target_lists,
    signal_line_list,
    trade_cost,
    balance,
    risk_per_trade,
    alpha_range,
    gamma_range,
    signal_line_if,
    signal_line_if_target,
    last_close,
)

i = 0
dff, check_index_first, check_index_second, check_index_third, check_index_fourth, base_histogram = dff_results_list[i]
dff.iloc[
    [
        check_index_first,
        check_index_second,
        check_index_third,
        check_index_fourth,
        base_histogram,
    ]
]

[np.float64(1431.0), np.float64(10.759398496240602), np.int64(0), np.int64(133), np.int64(45), np.int64(67), np.int64(26), np.float64(2825.5), np.int64(66), np.int64(19), np.float64(-1394.5), np.float64(2825.5), np.int64(67), np.int64(26), np.float64(2825.5), np.int64(67), np.int64(26)] 17
[np.float64(1431.0), np.float64(10.759398496240602), np.int64(0), np.int64(133), np.int64(45), np.int64(67), np.int64(26), np.float64(2825.5), np.int64(66), np.int64(19), np.float64(-1394.5), np.float64(2825.5), np.int64(67), np.int64(26), np.float64(2825.5), np.int64(67), np.int64(26)] 17
[np.float64(1431.0), np.float64(10.759398496240602), np.int64(0), np.int64(133), np.int64(45), np.int64(67), np.int64(26), np.float64(2825.5), np.int64(66), np.int64(19), np.float64(-1394.5), np.float64(2825.5), np.int64(67), np.int64(26), np.float64(2825.5), np.int64(67), np.int64(26)] 17
[np.float64(1431.0), np.float64(10.759398496240602), np.int64(0), np.int64(133), np.int64(45), np.int64(67), np.int64(26), np.f

,net,netbet,netpercent,bets,wins,longs,long_wins,long_profit,shorts,short_wins,short_profit,long_only_net,long_only_bets,long_only_wins,his
12,1431.0,10.759398,0,133,45,67,26,2825.5,66,19,-1394.5,2825.5,67,26,Histogram a: 0.3
12,1431.0,10.759398,0,133,45,67,26,2825.5,66,19,-1394.5,2825.5,67,26,Histogram a: 0.3
12,1431.0,10.759398,0,133,45,67,26,2825.5,66,19,-1394.5,2825.5,67,26,Histogram a: 0.3
12,1431.0,10.759398,0,133,45,67,26,2825.5,66,19,-1394.5,2825.5,67,26,Histogram a: 0.3
9,-3414.0,-19.734104,0,173,44,87,26,403.0,86,18,-3817.0,403.0,87,26,Histogram a: 0


In [8]:
best_sys1_major = extend_major_list(sys1_major)

[np.float64(-1615.0), np.float64(-9.252947182771743), np.int64(0), np.int64(441), np.int64(164), np.int64(221), np.int64(93), np.float64(5629.5), np.int64(220), np.int64(71), np.float64(-7244.5), np.float64(5629.5), np.int64(221), np.int64(93), np.float64(5629.5), np.int64(221), np.int64(93)] 17
[np.float64(6051.0), np.float64(67.72282387814424), np.int64(0), np.int64(299), np.int64(106), np.int64(150), np.int64(60), np.float64(9410.0), np.int64(149), np.int64(46), np.float64(-3359.0), np.float64(9410.0), np.int64(150), np.int64(60), np.float64(9410.0), np.int64(150), np.int64(60)] 17
[np.float64(6051.0), np.float64(67.72282387814424), np.int64(0), np.int64(299), np.int64(106), np.int64(150), np.int64(60), np.float64(9410.0), np.int64(149), np.int64(46), np.float64(-3359.0), np.float64(9410.0), np.int64(150), np.int64(60), np.float64(9410.0), np.int64(150), np.int64(60)] 17
[np.float64(-2690.0), np.float64(-22.764353516641847), np.int64(0), np.int64(333), np.int64(127), np.int64(167), 

# SYSTEM 2

In [9]:
signal_line_if = True
signal_line_if_target = False
sys2_major = [
    [ [0 for i in range(17)],
    [0 for i in range(17)],
    [0 for i in range(17)],
    [0 for i in range(17)],
    [0 for i in range(17)] ],
    [ [0 for i in range(17)],
    [0 for i in range(17)],
    [0 for i in range(17)],
    [0 for i in range(17)],
    [0 for i in range(17)] ],
]

# first part

In [10]:
(
    dff_results_list,
    
    sys2_major,
    sys2_first,
    best_sys2_first,
    
) = return_strategies_performance(
    nikkei_data,
    
    sys2_major,
    first_check,
    first_part,
    
    strategies,
    target_lists,
    signal_line_list,
    trade_cost,
    balance,
    risk_per_trade,
    alpha_range,
    gamma_range,
    signal_line_if,
    signal_line_if_target,
    last_close,
)

i = 0
dff, check_index_first, check_index_second, check_index_third, check_index_fourth, base_histogram = dff_results_list[i]
dff.iloc[
    [
        check_index_first,
        check_index_second,
        check_index_third,
        check_index_fourth,
        base_histogram,
    ]
]

[np.float64(-444.5), np.float64(-15.327586206896552), np.int64(0), np.int64(29), np.int64(21), np.int64(15), np.int64(11), np.float64(2254.5), np.int64(14), np.int64(10), np.float64(-2699.0), np.float64(2254.5), np.int64(15), np.int64(11), np.float64(2254.5), np.int64(15), np.int64(11)] 17
[np.float64(2684.5), np.float64(92.56896551724138), np.int64(0), np.int64(29), np.int64(21), np.int64(15), np.int64(11), np.float64(3868.5), np.int64(14), np.int64(10), np.float64(-1184.0), np.float64(3868.5), np.int64(15), np.int64(11), np.float64(3868.5), np.int64(15), np.int64(11)] 17
[np.float64(2684.5), np.float64(92.56896551724138), np.int64(0), np.int64(29), np.int64(21), np.int64(15), np.int64(11), np.float64(3868.5), np.int64(14), np.int64(10), np.float64(-1184.0), np.float64(3868.5), np.int64(15), np.int64(11), np.float64(3868.5), np.int64(15), np.int64(11)] 17
[np.float64(2684.5), np.float64(92.56896551724138), np.int64(0), np.int64(29), np.int64(21), np.int64(15), np.int64(11), np.float64

,net,netbet,netpercent,bets,wins,longs,long_wins,long_profit,shorts,short_wins,short_profit,long_only_net,long_only_bets,long_only_wins,his
10,-444.5,-15.327586,0,29,21,15,11,2254.5,14,10,-2699.0,2254.5,15,11,Histogram a: 0.1
16,2684.5,92.568966,0,29,21,15,11,3868.5,14,10,-1184.0,3868.5,15,11,Histogram a: 0.7
16,2684.5,92.568966,0,29,21,15,11,3868.5,14,10,-1184.0,3868.5,15,11,Histogram a: 0.7
16,2684.5,92.568966,0,29,21,15,11,3868.5,14,10,-1184.0,3868.5,15,11,Histogram a: 0.7
9,-152.5,-5.258621,0,29,20,15,11,2400.5,14,9,-2553.0,2400.5,15,11,Histogram a: 0


# second part

In [11]:
(
    dff_results_list,
    
    sys2_major,
    sys2_second,
    best_sys2_second,
    
) = return_strategies_performance(
    nikkei_data,
    
    sys2_major,
    second_check,
    second_part,
    
    strategies,
    target_lists,
    signal_line_list,
    trade_cost,
    balance,
    risk_per_trade,
    alpha_range,
    gamma_range,
    signal_line_if,
    signal_line_if_target,
    last_close,
)

i = 0
dff, check_index_first, check_index_second, check_index_third, check_index_fourth, base_histogram = dff_results_list[i]
dff.iloc[
    [
        check_index_first,
        check_index_second,
        check_index_third,
        check_index_fourth,
        base_histogram,
    ]
]

[np.float64(4806.5), np.float64(155.0483870967742), np.int64(0), np.int64(31), np.int64(23), np.int64(16), np.int64(12), np.float64(2898.0), np.int64(15), np.int64(11), np.float64(1908.5), np.float64(2898.0), np.int64(16), np.int64(12), np.float64(2898.0), np.int64(16), np.int64(12)] 17
[np.float64(8919.5), np.float64(371.6458333333333), np.int64(0), np.int64(24), np.int64(18), np.int64(12), np.int64(9), np.float64(5031.0), np.int64(12), np.int64(9), np.float64(3888.5), np.float64(5031.0), np.int64(12), np.int64(9), np.float64(5031.0), np.int64(12), np.int64(9)] 17
[np.float64(-2384.5), np.float64(-85.16071428571429), np.int64(0), np.int64(28), np.int64(16), np.int64(14), np.int64(9), np.float64(-781.0), np.int64(14), np.int64(7), np.float64(-1603.5), np.float64(-781.0), np.int64(14), np.int64(9), np.float64(-781.0), np.int64(14), np.int64(9)] 17
[np.float64(-2384.5), np.float64(-85.16071428571429), np.int64(0), np.int64(28), np.int64(16), np.int64(14), np.int64(9), np.float64(-781.0),

,net,netbet,netpercent,bets,wins,longs,long_wins,long_profit,shorts,short_wins,short_profit,long_only_net,long_only_bets,long_only_wins,his
8,4806.5,155.048387,0,31,23,16,12,2898.0,15,11,1908.5,2898.0,16,12,Histogram a: -0.1
17,8919.5,371.645833,0,24,18,12,9,5031.0,12,9,3888.5,5031.0,12,9,Histogram a: 0.8
307,-2384.5,-85.160714,0,28,16,14,9,-781.0,14,7,-1603.5,-781.0,14,9,Histogram a: 0.3 g: 1
307,-2384.5,-85.160714,0,28,16,14,9,-781.0,14,7,-1603.5,-781.0,14,9,Histogram a: 0.3 g: 1
9,3509.5,113.209677,0,31,23,16,12,2270.0,15,11,1239.5,2270.0,16,12,Histogram a: 0


# third part

In [12]:
(
    dff_results_list,
    
    sys2_major,
    sys2_third,
    best_sys2_third,
    
) = return_strategies_performance(
    nikkei_data,
    
    sys2_major,
    third_check,
    third_part,
    
    strategies,
    target_lists,
    signal_line_list,
    trade_cost,
    balance,
    risk_per_trade,
    alpha_range,
    gamma_range,
    signal_line_if,
    signal_line_if_target,
    last_close,
)

i = 0
dff, check_index_first, check_index_second, check_index_third, check_index_fourth, base_histogram = dff_results_list[i]
dff.iloc[
    [
        check_index_first,
        check_index_second,
        check_index_third,
        check_index_fourth,
        base_histogram,
    ]
]

[np.float64(5235.0), np.float64(180.51724137931035), np.int64(0), np.int64(29), np.int64(19), np.int64(15), np.int64(11), np.float64(4457.5), np.int64(14), np.int64(8), np.float64(777.5), np.float64(4457.5), np.int64(15), np.int64(11), np.float64(4457.5), np.int64(15), np.int64(11)] 17
[np.float64(3700.0), np.float64(127.58620689655173), np.int64(0), np.int64(29), np.int64(21), np.int64(15), np.int64(11), np.float64(3525.0), np.int64(14), np.int64(10), np.float64(175.0), np.float64(3525.0), np.int64(15), np.int64(11), np.float64(3525.0), np.int64(15), np.int64(11)] 17
[np.float64(4280.0), np.float64(142.66666666666666), np.int64(0), np.int64(30), np.int64(20), np.int64(15), np.int64(11), np.float64(3287.5), np.int64(15), np.int64(9), np.float64(992.5), np.float64(3287.5), np.int64(15), np.int64(11), np.float64(3287.5), np.int64(15), np.int64(11)] 17
[np.float64(4280.0), np.float64(142.66666666666666), np.int64(0), np.int64(30), np.int64(20), np.int64(15), np.int64(11), np.float64(3287.

,net,netbet,netpercent,bets,wins,longs,long_wins,long_profit,shorts,short_wins,short_profit,long_only_net,long_only_bets,long_only_wins,his
8,5235.0,180.517241,0,29,19,15,11,4457.5,14,8,777.5,4457.5,15,11,Histogram a: -0.1
1,3700.0,127.586207,0,29,21,15,11,3525.0,14,10,175.0,3525.0,15,11,Histogram a: -0.8
19,4280.0,142.666667,0,30,20,15,11,3287.5,15,9,992.5,3287.5,15,11,Histogram a: -0.9 g: 1
35,4280.0,142.666667,0,30,20,15,11,3287.5,15,9,992.5,3287.5,15,11,Histogram a: -0.9 g: 500
9,5535.0,190.862069,0,29,19,15,11,4607.5,14,8,927.5,4607.5,15,11,Histogram a: 0


In [13]:
best_sys2_major = extend_major_list(sys2_major)

[np.float64(9597.0), np.float64(320.238042269188), np.int64(0), np.int64(89), np.int64(63), np.int64(46), np.int64(34), np.float64(9610.0), np.int64(43), np.int64(29), np.float64(-13.0), np.float64(9610.0), np.int64(46), np.int64(34), np.float64(9610.0), np.int64(46), np.int64(34)] 17
[np.float64(15304.0), np.float64(591.8010057471264), np.int64(0), np.int64(82), np.int64(60), np.int64(42), np.int64(31), np.float64(12424.5), np.int64(40), np.int64(29), np.float64(2879.5), np.float64(12424.5), np.int64(42), np.int64(31), np.float64(12424.5), np.int64(42), np.int64(31)] 17
[np.float64(4580.0), np.float64(150.07491789819375), np.int64(0), np.int64(87), np.int64(57), np.int64(44), np.int64(31), np.float64(6375.0), np.int64(43), np.int64(26), np.float64(-1795.0), np.float64(6375.0), np.int64(44), np.int64(31), np.float64(6375.0), np.int64(44), np.int64(31)] 17
[np.float64(4580.0), np.float64(150.07491789819375), np.int64(0), np.int64(87), np.int64(57), np.int64(44), np.int64(31), np.float64

# SYSTEM 3

In [14]:
signal_line_if = False
signal_line_if_target = True
sys3_major = [
    [ [0 for i in range(17)],
    [0 for i in range(17)],
    [0 for i in range(17)],
    [0 for i in range(17)],
    [0 for i in range(17)] ],
    [ [0 for i in range(17)],
    [0 for i in range(17)],
    [0 for i in range(17)],
    [0 for i in range(17)],
    [0 for i in range(17)] ],
]

# first part

In [15]:
(
    dff_results_list,
    
    sys3_major,
    sys3_first,
    best_sys3_first,
    
) = return_strategies_performance(
    nikkei_data,
    
    sys3_major,
    first_check,
    first_part,
    
    strategies,
    target_lists,
    signal_line_list,
    trade_cost,
    balance,
    risk_per_trade,
    alpha_range,
    gamma_range,
    signal_line_if,
    signal_line_if_target,
    last_close,
)

i = 0
dff, check_index_first, check_index_second, check_index_third, check_index_fourth, base_histogram = dff_results_list[i]
dff.iloc[
    [
        check_index_first,
        check_index_second,
        check_index_third,
        check_index_fourth,
        base_histogram,
    ]
]

[np.float64(-444.5), np.float64(-15.327586206896552), np.int64(0), np.int64(29), np.int64(21), np.int64(15), np.int64(11), np.float64(2254.5), np.int64(14), np.int64(10), np.float64(-2699.0), np.float64(2254.5), np.int64(15), np.int64(11), np.float64(2254.5), np.int64(15), np.int64(11)] 17
[np.float64(2684.5), np.float64(92.56896551724138), np.int64(0), np.int64(29), np.int64(21), np.int64(15), np.int64(11), np.float64(3868.5), np.int64(14), np.int64(10), np.float64(-1184.0), np.float64(3868.5), np.int64(15), np.int64(11), np.float64(3868.5), np.int64(15), np.int64(11)] 17
[np.float64(2684.5), np.float64(92.56896551724138), np.int64(0), np.int64(29), np.int64(21), np.int64(15), np.int64(11), np.float64(3868.5), np.int64(14), np.int64(10), np.float64(-1184.0), np.float64(3868.5), np.int64(15), np.int64(11), np.float64(3868.5), np.int64(15), np.int64(11)] 17
[np.float64(2209.5), np.float64(76.1896551724138), np.int64(0), np.int64(29), np.int64(18), np.int64(15), np.int64(10), np.float64(

,net,netbet,netpercent,bets,wins,longs,long_wins,long_profit,shorts,short_wins,short_profit,long_only_net,long_only_bets,long_only_wins,his
10,-444.5,-15.327586,0,29,21,15,11,2254.5,14,10,-2699.0,2254.5,15,11,Histogram a: 0.1
16,2684.5,92.568966,0,29,21,15,11,3868.5,14,10,-1184.0,3868.5,15,11,Histogram a: 0.7
16,2684.5,92.568966,0,29,21,15,11,3868.5,14,10,-1184.0,3868.5,15,11,Histogram a: 0.7
411,2209.5,76.189655,0,29,18,15,10,3497.5,14,8,-1288.0,3497.5,15,10,Histogram a: 0.7 g: 60
9,-152.5,-5.258621,0,29,20,15,11,2400.5,14,9,-2553.0,2400.5,15,11,Histogram a: 0


# second part

In [16]:
(
    dff_results_list,
    
    sys3_major,
    sys3_second,
    best_sys3_second,
    
) = return_strategies_performance(
    nikkei_data,
    
    sys3_major,
    second_check,
    second_part,
    
    strategies,
    target_lists,
    signal_line_list,
    trade_cost,
    balance,
    risk_per_trade,
    alpha_range,
    gamma_range,
    signal_line_if,
    signal_line_if_target,
    last_close,
)

i = 0
dff, check_index_first, check_index_second, check_index_third, check_index_fourth, base_histogram = dff_results_list[i]
dff.iloc[
    [
        check_index_first,
        check_index_second,
        check_index_third,
        check_index_fourth,
        base_histogram,
    ]
]

[np.float64(4806.5), np.float64(155.0483870967742), np.int64(0), np.int64(31), np.int64(23), np.int64(16), np.int64(12), np.float64(2898.0), np.int64(15), np.int64(11), np.float64(1908.5), np.float64(2898.0), np.int64(16), np.int64(12), np.float64(2898.0), np.int64(16), np.int64(12)] 17
[np.float64(8982.0), np.float64(408.27272727272725), np.int64(0), np.int64(22), np.int64(13), np.int64(11), np.int64(8), np.float64(5025.0), np.int64(11), np.int64(5), np.float64(3957.0), np.float64(5025.0), np.int64(11), np.int64(8), np.float64(5025.0), np.int64(11), np.int64(8)] 17
[np.float64(8982.0), np.float64(408.27272727272725), np.int64(0), np.int64(22), np.int64(13), np.int64(11), np.int64(8), np.float64(5025.0), np.int64(11), np.int64(5), np.float64(3957.0), np.float64(5025.0), np.int64(11), np.int64(8), np.float64(5025.0), np.int64(11), np.int64(8)] 17
[np.float64(8982.0), np.float64(408.27272727272725), np.int64(0), np.int64(22), np.int64(13), np.int64(11), np.int64(8), np.float64(5025.0), n

,net,netbet,netpercent,bets,wins,longs,long_wins,long_profit,shorts,short_wins,short_profit,long_only_net,long_only_bets,long_only_wins,his
8,4806.5,155.048387,0,31,23,16,12,2898.0,15,11,1908.5,2898.0,16,12,Histogram a: -0.1
451,8982.0,408.272727,0,22,13,11,8,5025.0,11,5,3957.0,5025.0,11,8,Histogram a: 0.9 g: 1
451,8982.0,408.272727,0,22,13,11,8,5025.0,11,5,3957.0,5025.0,11,8,Histogram a: 0.9 g: 1
463,8982.0,408.272727,0,22,13,11,8,5025.0,11,5,3957.0,5025.0,11,8,Histogram a: 0.9 g: 100
9,3509.5,113.209677,0,31,23,16,12,2270.0,15,11,1239.5,2270.0,16,12,Histogram a: 0


# third part

In [17]:
(
    dff_results_list,
    
    sys3_major,
    sys3_third,
    best_sys3_third,
    
) = return_strategies_performance(
    nikkei_data,
    
    sys3_major,
    third_check,
    third_part,
    
    strategies,
    target_lists,
    signal_line_list,
    trade_cost,
    balance,
    risk_per_trade,
    alpha_range,
    gamma_range,
    signal_line_if,
    signal_line_if_target,
    last_close,
)

i = 0
dff, check_index_first, check_index_second, check_index_third, check_index_fourth, base_histogram = dff_results_list[i]
dff.iloc[
    [
        check_index_first,
        check_index_second,
        check_index_third,
        check_index_fourth,
        base_histogram,
    ]
]

[np.float64(5235.0), np.float64(180.51724137931035), np.int64(0), np.int64(29), np.int64(19), np.int64(15), np.int64(11), np.float64(4457.5), np.int64(14), np.int64(8), np.float64(777.5), np.float64(4457.5), np.int64(15), np.int64(11), np.float64(4457.5), np.int64(15), np.int64(11)] 17
[np.float64(4965.0), np.float64(171.20689655172413), np.int64(0), np.int64(29), np.int64(21), np.int64(15), np.int64(11), np.float64(4175.0), np.int64(14), np.int64(10), np.float64(790.0), np.float64(4175.0), np.int64(15), np.int64(11), np.float64(4175.0), np.int64(15), np.int64(11)] 17
[np.float64(4380.0), np.float64(292.0), np.int64(0), np.int64(15), np.int64(12), np.int64(8), np.int64(6), np.float64(3435.0), np.int64(7), np.int64(6), np.float64(945.0), np.float64(3435.0), np.int64(8), np.int64(6), np.float64(3435.0), np.int64(8), np.int64(6)] 17
[np.float64(4380.0), np.float64(292.0), np.int64(0), np.int64(15), np.int64(12), np.int64(8), np.int64(6), np.float64(3435.0), np.int64(7), np.int64(6), np.fl

,net,netbet,netpercent,bets,wins,longs,long_wins,long_profit,shorts,short_wins,short_profit,long_only_net,long_only_bets,long_only_wins,his
8,5235.0,180.517241,0,29,19,15,11,4457.5,14,8,777.5,4457.5,15,11,Histogram a: -0.1
0,4965.0,171.206897,0,29,21,15,11,4175.0,14,10,790.0,4175.0,15,11,Histogram a: -0.9
451,4380.0,292.000000,0,15,12,8,6,3435.0,7,6,945.0,3435.0,8,6,Histogram a: 0.9 g: 1
470,4380.0,292.000000,0,15,12,8,6,3435.0,7,6,945.0,3435.0,8,6,Histogram a: 0.9 g: 800
9,5535.0,190.862069,0,29,19,15,11,4607.5,14,8,927.5,4607.5,15,11,Histogram a: 0


In [18]:
best_sys3_major = extend_major_list(sys3_major)

[np.float64(9597.0), np.float64(320.238042269188), np.int64(0), np.int64(89), np.int64(63), np.int64(46), np.int64(34), np.float64(9610.0), np.int64(43), np.int64(29), np.float64(-13.0), np.float64(9610.0), np.int64(46), np.int64(34), np.float64(9610.0), np.int64(46), np.int64(34)] 17
[np.float64(16631.5), np.float64(672.0485893416928), np.int64(0), np.int64(80), np.int64(55), np.int64(41), np.int64(30), np.float64(13068.5), np.int64(39), np.int64(25), np.float64(3563.0), np.float64(13068.5), np.int64(41), np.int64(30), np.float64(13068.5), np.int64(41), np.int64(30)] 17
[np.float64(16046.5), np.float64(792.8416927899686), np.int64(0), np.int64(66), np.int64(46), np.int64(34), np.int64(25), np.float64(12328.5), np.int64(32), np.int64(21), np.float64(3718.0), np.float64(12328.5), np.int64(34), np.int64(25), np.float64(12328.5), np.int64(34), np.int64(25)] 17
[np.float64(15571.5), np.float64(776.462382445141), np.int64(0), np.int64(66), np.int64(43), np.int64(34), np.int64(24), np.float6

# tables

In [ ]:
crossover_modes = ["MACD Crossover", "Zero Crossover"]
sys_modes = ["System 1", "System 2", "System 3"]
entries = ["Rule 1", "Rule 2", "Rule 3", "GA", "MACD"]
tuples = []

data = [sys1_first[0], sys2_first[0], sys3_first[0],
        sys1_first[1], sys2_first[1], sys3_first[1]]
# data = [sys1_major_macd_crossover, sys2_major_macd_crossover]

for crossover_mode in crossover_modes:
    for sys_mode in sys_modes:
        for entry in entries:
            tuples.append((crossover_mode, sys_mode, entry))
table_index = pd.MultiIndex.from_tuples(tuples, names=["first", "second", "third"])

table_data = table_data_generator(data, table_index, 5)

columns = [
    "Return",
    "Win Ratio",
    "Long Return",
    "Long Win Ratio",
    "Short Return",
    "Short Win Ratio",
    # "Long 1 Return",
    # "Long 1 Win Ratio",
    # "Long 2 Return",
    # "Long 2 Win Ratio",
]
for col in columns:
    l = []
    for x in table_data[col]:
        l.append(round(x, 2))
    table_data[col] = l

table_data.to_csv(f"./results/nikkei/{params_str}/nikkei_rules_first_part.csv", index=True)
caption = f"Results for {macd_params} paramters for MACD by applying the normal scenario on first part of \\citet{"{jrfm14010037}"}'s data"
write_table_tex(table_data, f"./results/nikkei/{params_str}", "nikkei_rules_first_part", f"nikkei_{params_str}_first", caption)
write_table_tex(table_data, prefix_tex_tables, f"nikkei_rules_first_part_{params_str}", f"nikkei_{params_str}_first", caption)
text = f"\\input{"{"}nikkei_rules_first_part_{params_str}_tex{"}"}"
if macd_params == (12, 26, 9):
    write_to_text(prefix_text, file_name, text, mode='w')
else:
    write_to_text(prefix_text, file_name, text, mode='a')
    
table_data

Return  Transactions  Win Ratio  Long Return  \
first          second   third                                                  
MACD Crossover System 1 Rule 1 -1788.5           146      32.19       1786.5   
                        Rule 2  2068.5            92      39.13       3662.5   
                        Rule 3  2068.5            92      39.13       3662.5   
                        GA      2877.5            92      46.74       4141.5   
                        MACD   -2150.5           156      31.41       1605.5   
               System 2 Rule 1  -444.5            29      72.41       2254.5   
                        Rule 2  2684.5            29      72.41       3868.5   
                        Rule 3  2684.5            29      72.41       3868.5   
                        GA      2684.5            29      72.41       3868.5   
                        MACD    -152.5            29      68.97       2400.5   
               System 3 Rule 1  -444.5            29      72.41       2254.5   
                        Rule 2  2684.5            29      72.41       3868.5   
                        Rule 3  2684.5            29      72.41       3868.5   
                        GA      2209.5            29      62.07       3497.5   
                        MACD    -152.5            29      68.97       2400.5   
Zero Crossover System 1 Rule 1  4164.5            53      26.42       4222.5   
                        Rule 2  2113.5           119      33.61       3847.5   
                        Rule 3  2113.5           119      33.61       3847.5   
                        GA      1501.5            35      34.29       3154.5   
                        MACD    4353.5            37      32.43       4364.5   
               System 2 Rule 1 -2815.5            18      61.11        625.0   
                        Rule 2 -4491.5             7      42.86        103.0   
                        Rule 3 -2997.5            34      70.59        534.0   
                        GA     -2997.5            34      70.59        534.0   
                        MACD    4885.5            23      43.48       4530.5   
               System 3 Rule 1  4900.5            25      36.00       4485.5   
                        Rule 2  7131.5            19      52.63       5648.5   
                        Rule 3  7131.5            19      52.63       5648.5   
                        GA      6038.5            19      36.84       5054.5   
                        MACD    4885.5            23      43.48       4530.5   

                                Long Transactions  Long Win Ratio  \
first          second   third                                       
MACD Crossover System 1 Rule 1                 73           39.73   
                        Rule 2                 46           45.65   
                        Rule 3                 46           45.65   
                        GA                     46           52.17   
                        MACD                   78           37.18   
               System 2 Rule 1                 15           73.33   
                        Rule 2                 15           73.33   
                        Rule 3                 15           73.33   
                        GA                     15           73.33   
                        MACD                   15           73.33   
               System 3 Rule 1                 15           73.33   
                        Rule 2                 15           73.33   
                        Rule 3                 15           73.33   
                        GA                     15           66.67   
                        MACD                   15           73.33   
Zero Crossover System 1 Rule 1                 26           26.92   
                        Rule 2                 59           38.98   
                        Rule 3                 59           38.98   
                        GA                     17           41.18   
                      

In [ ]:
crossover_modes = ["MACD Crossover", "Zero Crossover"]
sys_modes = ["System 1", "System 2", "System 3"]
entries = ["Rule 1", "Rule 2", "Rule 3", "GA", "MACD"]
tuples = []

data = [sys1_second[0], sys2_second[0], sys3_second[0],
        sys1_second[1], sys2_second[1], sys3_second[1]]
# data = [sys1_major_macd_crossover, sys2_major_macd_crossover]

for crossover_mode in crossover_modes:
    for sys_mode in sys_modes:
        for entry in entries:
            tuples.append((crossover_mode, sys_mode, entry))
table_index = pd.MultiIndex.from_tuples(tuples, names=["first", "second", "third"])

table_data = table_data_generator(data, table_index, 5)

columns = [
    "Return",
    "Win Ratio",
    "Long Return",
    "Long Win Ratio",
    "Short Return",
    "Short Win Ratio",
    # "Long 1 Return",
    # "Long 1 Win Ratio",
    # "Long 2 Return",
    # "Long 2 Win Ratio",
]
for col in columns:
    l = []
    for x in table_data[col]:
        l.append(round(x, 2))
    table_data[col] = l

table_data.to_csv(f"./results/nikkei/{params_str}/nikkei_rules_second_part.csv", index=True)
caption = f"Results for {macd_params} paramters for MACD by applying the normal scenario on second part of \\citet{"{jrfm14010037}"}'s data"
write_table_tex(table_data, f"./results/nikkei/{params_str}", "nikkei_rules_second_part", f"nikkei_{params_str}_second", caption)
write_table_tex(table_data, prefix_tex_tables, f"nikkei_rules_second_part_{params_str}", f"nikkei_{params_str}_second", caption)
text = f"\\input{"{"}nikkei_rules_second_part_{params_str}_tex{"}"}"
write_to_text(prefix_text, file_name, text, mode=mode)
table_data

Return  Transactions  Win Ratio  Long Return  \
first          second   third                                                  
MACD Crossover System 1 Rule 1 -1257.5           162      44.44       1017.5   
                        Rule 2  2551.5            74      33.78       2922.0   
                        Rule 3  2551.5            74      33.78       2922.0   
                        GA     -6998.5           108      36.11      -1853.0   
                        MACD    2082.5           153      36.60       2856.5   
               System 2 Rule 1  4806.5            31      74.19       2898.0   
                        Rule 2  8919.5            24      75.00       5031.0   
                        Rule 3 -2384.5            28      57.14       -781.0   
                        GA     -2384.5            28      57.14       -781.0   
                        MACD    3509.5            31      74.19       2270.0   
               System 3 Rule 1  4806.5            31      74.19       2898.0   
                        Rule 2  8982.0            22      59.09       5025.0   
                        Rule 3  8982.0            22      59.09       5025.0   
                        GA      8982.0            22      59.09       5025.0   
                        MACD    3509.5            31      74.19       2270.0   
Zero Crossover System 1 Rule 1 -5890.5            55      23.64      -2074.5   
                        Rule 2 -4861.5            57      24.56      -1560.0   
                        Rule 3 -4654.5           101      40.59      -1487.0   
                        GA     -4654.5           101      40.59      -1487.0   
                        MACD   -6330.5            55      21.82      -2294.5   
               System 2 Rule 1 -2848.5            30      33.33       -718.5   
                        Rule 2  4415.5            22      63.64       3179.5   
                        Rule 3  4415.5            22      63.64       3179.5   
                        GA      4415.5            22      63.64       3179.5   
                        MACD   -5143.5            22      22.73      -1866.0   
               System 3 Rule 1  -605.5            20      35.00        -34.5   
                        Rule 2  5599.5            22      54.55       4151.5   
                        Rule 3  5599.5            22      54.55       4151.5   
                        GA      5599.5            22      54.55       4151.5   
                        MACD   -5143.5            22      22.73      -1866.0   

                                Long Transactions  Long Win Ratio  \
first          second   third                                       
MACD Crossover System 1 Rule 1                 81           46.91   
                        Rule 2                 37           35.14   
                        Rule 3                 37           35.14   
                        GA                     54           37.04   
                        MACD                   77           41.56   
               System 2 Rule 1                 16           75.00   
                        Rule 2                 12           75.00   
                        Rule 3                 14           64.29   
                        GA                     14           64.29   
                        MACD                   16           75.00   
               System 3 Rule 1                 16           75.00   
                        Rule 2                 11           72.73   
                        Rule 3                 11           72.73   
                        GA                     11           72.73   
                        MACD                   16           75.00   
Zero Crossover System 1 Rule 1                 27           29.63   
                        Rule 2                 28           28.57   
                        Rule 3                 50           40.00   
                        GA                     50           40.00   
                      

In [ ]:
crossover_modes = ["MACD Crossover", "Zero Crossover"]
sys_modes = ["System 1", "System 2", "System 3"]
entries = ["Rule 1", "Rule 2", "Rule 3", "GA", "MACD"]
tuples = []

data = [sys1_third[0], sys2_third[0], sys3_third[0],
        sys1_third[1], sys2_third[1], sys3_third[1]]
# data = [sys1_major_macd_crossover, sys2_major_macd_crossover]

for crossover_mode in crossover_modes:
    for sys_mode in sys_modes:
        for entry in entries:
            tuples.append((crossover_mode, sys_mode, entry))
table_index = pd.MultiIndex.from_tuples(tuples, names=["first", "second", "third"])

table_data = table_data_generator(data, table_index, 5)

columns = [
    "Return",
    "Win Ratio",
    "Long Return",
    "Long Win Ratio",
    "Short Return",
    "Short Win Ratio",
    # "Long 1 Return",
    # "Long 1 Win Ratio",
    # "Long 2 Return",
    # "Long 2 Win Ratio",
]
for col in columns:
    l = []
    for x in table_data[col]:
        l.append(round(x, 2))
    table_data[col] = l

table_data.to_csv(f"./results/nikkei/{params_str}/nikkei_rules_third_part.csv", index=True)
caption = f"Results for {macd_params} paramters for MACD by applying the normal scenario on third part of \\citet{"{jrfm14010037}"}'s data"
write_table_tex(table_data, f"./results/nikkei/{params_str}", "nikkei_rules_third_part", f"nikkei_{params_str}_third", caption)
write_table_tex(table_data, prefix_tex_tables, f"nikkei_rules_third_part_{params_str}", f"nikkei_{params_str}_third", caption)
text = f"\\input{"{"}nikkei_rules_third_part_{params_str}_tex{"}"}"
write_to_text(prefix_text, file_name, text, mode=mode)
table_data

Return  Transactions  Win Ratio  Long Return  \
first          second   third                                                  
MACD Crossover System 1 Rule 1  1431.0           133      33.83       2825.5   
                        Rule 2  1431.0           133      33.83       2825.5   
                        Rule 3  1431.0           133      33.83       2825.5   
                        GA      1431.0           133      33.83       2825.5   
                        MACD   -3414.0           173      25.43        403.0   
               System 2 Rule 1  5235.0            29      65.52       4457.5   
                        Rule 2  3700.0            29      72.41       3525.0   
                        Rule 3  4280.0            30      66.67       3287.5   
                        GA      4280.0            30      66.67       3287.5   
                        MACD    5535.0            29      65.52       4607.5   
               System 3 Rule 1  5235.0            29      65.52       4457.5   
                        Rule 2  4965.0            29      72.41       4175.0   
                        Rule 3  4380.0            15      80.00       3435.0   
                        GA      4380.0            15      80.00       3435.0   
                        MACD    5535.0            29      65.52       4607.5   
Zero Crossover System 1 Rule 1 -4850.0            43      27.91       -812.5   
                        Rule 2 -3220.0            41      24.39       -155.0   
                        Rule 3 -5190.0            53      32.08       -960.0   
                        GA     -5190.0            53      32.08       -960.0   
                        MACD      95.0            48      29.17       2252.5   
               System 2 Rule 1  4450.0            12      75.00       3240.0   
                        Rule 2 -1050.0             8      25.00      -1565.0   
                        Rule 3  4725.0            10      70.00       3377.5   
                        GA      4725.0            10      70.00       3377.5   
                        MACD    6550.0            18      50.00       4442.5   
               System 3 Rule 1  4292.5            16      50.00       3325.0   
                        Rule 2 -1385.0            24      29.17        590.0   
                        Rule 3 -1385.0            24      29.17        590.0   
                        GA      1475.0            22      36.36       3015.0   
                        MACD    6550.0            18      50.00       4442.5   

                                Long Transactions  Long Win Ratio  \
first          second   third                                       
MACD Crossover System 1 Rule 1                 67           38.81   
                        Rule 2                 67           38.81   
                        Rule 3                 67           38.81   
                        GA                     67           38.81   
                        MACD                   87           29.89   
               System 2 Rule 1                 15           73.33   
                        Rule 2                 15           73.33   
                        Rule 3                 15           73.33   
                        GA                     15           73.33   
                        MACD                   15           73.33   
               System 3 Rule 1                 15           73.33   
                        Rule 2                 15           73.33   
                        Rule 3                  8           75.00   
                        GA                      8           75.00   
                        MACD                   15           73.33   
Zero Crossover System 1 Rule 1                 21           28.57   
                        Rule 2                 20           25.00   
                        Rule 3                 26           34.62   
                        GA                     26           34.62   
                      

In [ ]:
crossover_modes = ["MACD Crossover", "Zero Crossover"]
sys_modes = ["System 1", "System 2", "System 3"]
entries = ["Rule 1", "Rule 2", "Rule 3", "GA", "MACD"]
tuples = []

data = [sys1_major[0], sys2_major[0], sys3_major[0],
        sys1_major[1], sys2_major[1], sys3_major[1]]
# data = [sys1_major_macd_crossover, sys2_major_macd_crossover]

for crossover_mode in crossover_modes:
    for sys_mode in sys_modes:
        for entry in entries:
            tuples.append((crossover_mode, sys_mode, entry))
table_index = pd.MultiIndex.from_tuples(tuples, names=["first", "second", "third"])

table_data = table_data_generator(data, table_index, 5)

columns = [
    "Return",
    "Win Ratio",
    "Long Return",
    "Long Win Ratio",
    "Short Return",
    "Short Win Ratio",
    # "Long 1 Return",
    # "Long 1 Win Ratio",
    # "Long 2 Return",
    # "Long 2 Win Ratio",
]
for col in columns:
    l = []
    for x in table_data[col]:
        l.append(round(x, 2))
    table_data[col] = l

table_data.to_csv(f"./results/nikkei/{params_str}/nikkei_rules_major.csv", index=True)
caption = f"Results for {macd_params} paramters for MACD by applying the normal scenario on \\citet{"{jrfm14010037}"}'s data"
write_table_tex(table_data, f"./results/nikkei/{params_str}", "nikkei_rules_major", f"nikkei_{params_str}_major", caption)
write_table_tex(table_data, prefix_tex_tables, f"nikkei_rules_major_{params_str}", f"nikkei_{params_str}_major", caption)
text = f"\\input{"{"}nikkei_rules_major_{params_str}_tex{"}"}"
write_to_text(prefix_text, file_name, text, mode=mode)
table_data

Return  Transactions  Win Ratio  Long Return  \
first          second   third                                                   
MACD Crossover System 1 Rule 1  -1615.0           441      37.19       5629.5   
                        Rule 2   6051.0           299      35.45       9410.0   
                        Rule 3   6051.0           299      35.45       9410.0   
                        GA      -2690.0           333      38.14       5114.0   
                        MACD    -3482.0           482      30.91       4865.0   
               System 2 Rule 1   9597.0            89      70.79       9610.0   
                        Rule 2  15304.0            82      73.17      12424.5   
                        Rule 3   4580.0            87      65.52       6375.0   
                        GA       4580.0            87      65.52       6375.0   
                        MACD     8892.0            89      69.66       9278.0   
               System 3 Rule 1   9597.0            89      70.79       9610.0   
                        Rule 2  16631.5            80      68.75      13068.5   
                        Rule 3  16046.5            66      69.70      12328.5   
                        GA      15571.5            66      65.15      11957.5   
                        MACD     8892.0            89      69.66       9278.0   
Zero Crossover System 1 Rule 1  -6576.0           151      25.83       1335.5   
                        Rule 2  -5968.0           217      29.49       2132.5   
                        Rule 3  -7731.0           273      35.90       1400.5   
                        GA      -8343.0           189      37.04        707.5   
                        MACD    -1882.0           140      27.14       4322.5   
               System 2 Rule 1  -1214.0            60      50.00       3146.5   
                        Rule 2  -1126.0            37      51.35       1717.5   
                        Rule 3   6143.0            66      68.18       7091.0   
                        GA       6143.0            66      68.18       7091.0   
                        MACD     6292.0            63      38.10       7107.0   
               System 3 Rule 1   8587.5            61      39.34       7776.0   
                        Rule 2  11346.0            65      44.62      10390.0   
                        Rule 3  11346.0            65      44.62      10390.0   
                        GA      13113.0            63      42.86      12221.0   
                        MACD     6292.0            63      38.10       7107.0   

                                Long Transactions  Long Win Ratio  \
first          second   third                                       
MACD Crossover System 1 Rule 1                221           42.08   
                        Rule 2                150           40.00   
                        Rule 3                150           40.00   
                        GA                    167           41.92   
                        MACD                  242           35.95   
               System 2 Rule 1                 46           73.91   
                        Rule 2                 42           73.81   
                        Rule 3                 44           70.45   
                        GA                     44           70.45   
                        MACD                   46           73.91   
               System 3 Rule 1                 46           73.91   
                        Rule 2                 41           73.17   
                        Rule 3                 34           73.53   
                        GA                     34           70.59   
                        MACD                   46           73.91   
Zero Crossover System 1 Rule 1                 74           28.38   
                        Rule 2                107           33.64   
                        Rule 3                135           38.52   
                        GA                     93           

In [23]:
crossover_modes = ["MACD Crossover", "Zero Crossover"]
sys_modes = ["System 1", "System 2"]
entries = ["Best", "MACD"]
tuples = []

# data = [[best_sys1_major_macd_crossover, sys1_major_macd_crossover[-1]], [best_sys2_major_macd_crossover, sys2_major_macd_crossover[-1]], [best_sys3_major_macd_crossover, sys3_major_macd_crossover[-1]]]
data = [
    [best_sys1_major[0], sys1_major[0][-1]],
    [best_sys2_major[0], sys2_major[0][-1]],
    [best_sys1_major[1], sys1_major[1][-1]],
    [best_sys2_major[1], sys2_major[1][-1]],
]

for crossover_mode in crossover_modes:
    for sys_mode in sys_modes:
        for entry in entries:
            tuples.append((crossover_mode, sys_mode, entry))
table_index = pd.MultiIndex.from_tuples(tuples, names=["first", "second", "third"])

table_data = table_data_generator(data, table_index, 2)

columns = [
    "Return",
    "Win Ratio",
    "Long Return",
    "Long Win Ratio",
    "Short Return",
    "Short Win Ratio",
    # "Long 1 Return",
    # "Long 1 Win Ratio",
    # "Long 2 Return",
    # "Long 2 Win Ratio",
]
for col in columns:
    l = []
    for x in table_data[col]:
        l.append(round(x, 2))
    table_data[col] = l

table_data.to_csv(
    f"./results/nikkei/{params_str}/nikkei_rules_overall_major.csv", index=True
)


# The csv keeps all fourteen columns. The tex table takes a chosen subset, because
# "Long 1" is the long LEG of the long/short strategy and is structurally the same as
# "Long Return" for an always-in-market crossover system, so printing both would just
# repeat a column. "Long 2" is the separate long-only strategy and is the one worth
# showing. The first eight must stay in this order: the published comparison rows line
# up against them.
tex_columns = [
    "Return",
    "Transactions",
    "Win Ratio",
    "Long Return",
    "Long Transactions",
    "Long Win Ratio",
    "Short Return",
    "Short Win Ratio",
    "Long 2 Return",
    "Long 2 Transactions",
    "Long 2 Win Ratio",
]
tex_data = table_data[tex_columns].copy()

if macd_params == (12, 26, 9):
    # the caption has to carry two things the numbers alone do not say: which of our two
    # crossover blocks corresponds to his rule, and that our series is the index while his
    # was front-month futures
    caption = (
        "Comparison between the results published in \\citet{jrfm14010037} and the results "
        "obtained by applying the normal scenario on their data. \\citet{jrfm14010037} uses the "
        "signal line crossover as his trading rule, which corresponds to our MACD Crossover "
        "rows; our Zero Crossover rows are reported in addition and have no counterpart in "
        "his study. His series is the Nikkei 225 front-month futures contract from JPX Data "
        "Cloud, whereas ours is the Nikkei 225 index, so basis and quarterly roll are not "
        "replicated."
    )
    write_overall_table_tex(tex_data, f"./results/nikkei/{params_str}", "nikkei_rules_overall_major", f"nikkei_{params_str}_overall_major", caption, "kang")
    write_overall_table_tex(tex_data, prefix_tex_tables, f"nikkei_rules_overall_major_{params_str}", f"nikkei_{params_str}_overall_major", caption, "kang")
else:
    write_overall_table_tex(tex_data, f"./results/nikkei/{params_str}", "nikkei_rules_overall_major", f"nikkei_{params_str}_overall_major", caption, None)
    write_overall_table_tex(tex_data, prefix_tex_tables, f"nikkei_rules_overall_major_{params_str}", f"nikkei_{params_str}_overall_major", caption, None)
text = f"\\input{"{"}nikkei_rules_overall_major_{params_str}_tex{"}"}"
write_to_text(prefix_text, file_name, text, mode=mode)
table_data

Return  Transactions  Win Ratio  Long Return  \
first          second   third                                                  
MACD Crossover System 1 Best    6051.0           299      35.45       9410.0   
                        MACD   -3482.0           482      30.91       4865.0   
               System 2 Best   15304.0            82      73.17      12424.5   
                        MACD    8892.0            89      69.66       9278.0   
Zero Crossover System 1 Best   -5968.0           217      29.49       2132.5   
                        MACD   -1882.0           140      27.14       4322.5   
               System 2 Best    6143.0            66      68.18       7091.0   
                        MACD    6292.0            63      38.10       7107.0   

                               Long Transactions  Long Win Ratio  \
first          second   third                                      
MACD Crossover System 1 Best                 150           40.00   
                        MACD                 242           35.95   
               System 2 Best                  42           73.81   
                        MACD                  46           73.91   
Zero Crossover System 1 Best                 107           33.64   
                        MACD                  69           28.99   
               System 2 Best                  33           75.76   
                        MACD                  31           45.16   

                               Short Return  Short Win Ratio  
first          second   third                                 
MACD Crossover System 1 Best        -3359.0            30.87  
                        MACD        -8347.0            25.83  
               System 2 Best         2879.5            72.50  
                        MACD         -386.0            65.12  
Zero Crossover System 1 Best        -8100.5            25.45  
                        MACD        -6204.5            25.35  
               System 2 Best         -948.0            60.61  
                        MACD         -815.0            31.25